# EcoMarket - Atención al Cliente con RAG (Ollama + LangChain)

Este notebook implementa un sistema de **Retrieval-Augmented Generation (RAG)** para el caso **EcoMarket**, un asistente virtual de atención al cliente. El sistema utiliza:

- **[Ollama](https://ollama.com)** como servidor local del LLM (`llama3.2:3b`).
- **[LangChain](https://www.langchain.com)** como framework para orquestar el pipeline RAG.
- **FAISS** como vectorstore para la indexación y recuperación de documentos.
- **HuggingFace embeddings** multilingües para representar los textos.

### Fuentes de conocimiento
El asistente de EcoMarket responde consultas a partir de dos fuentes locales:

1. `data/pedidos.json` — Estado y detalle de los pedidos de los clientes.
2. `data/politica_devoluciones.txt` — Política oficial de devoluciones de EcoMarket.

Los prompts usados por el modelo se cargan desde la carpeta `prompts/`:
- `prompts/prompt_pedido.txt`
- `prompts/prompt_devolucion.txt`


In [ ]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

In [ ]:
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 lshw -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install langchain-ollama langchain-community langchain-huggingface ollama faiss-gpu-cu12 sentence-transformers gradio colab-xterm

## 1. Carga de documentos locales

Cargamos las dos fuentes locales y las convertimos en `Document` de LangChain. Cada documento lleva metadatos (`source`, `tipo`) para poder citarlo en las respuestas.

In [ ]:
import json
from pathlib import Path
from langchain.schema import Document

DATA_DIR = Path('data')
PEDIDOS_PATH = DATA_DIR / 'pedidos.json'
POLITICA_PATH = DATA_DIR / 'politica_devoluciones.txt'

def cargar_pedidos(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        pedidos = json.load(f)
    docs = []
    for p in pedidos:
        contenido = (
            f"Pedido {p['tracking_number']}\n"
            f"Estado: {p['estado']}\n"
            f"Fecha estimada de entrega: {p['fecha_entrega']}\n"
            f"Detalle: {p['detalle']}"
        )
        docs.append(Document(
            page_content=contenido,
            metadata={
                'source': str(path),
                'tipo': 'pedido',
                'tracking_number': p['tracking_number'],
            },
        ))
    return docs

def cargar_politica(path: Path):
    texto = path.read_text(encoding='utf-8')
    return [Document(
        page_content=texto,
        metadata={'source': str(path), 'tipo': 'politica_devoluciones'},
    )]

docs_pedidos = cargar_pedidos(PEDIDOS_PATH)
docs_politica = cargar_politica(POLITICA_PATH)
documents = docs_pedidos + docs_politica

print(f"Pedidos cargados: {len(docs_pedidos)}")
print(f"Documentos de política: {len(docs_politica)}")
print(f"Total de documentos: {len(documents)}")

## 2. Instalación y arranque de Ollama

Si Ollama no está instalado, lo instalamos. Luego descargamos el modelo `llama3.2:3b`. En Colab, debes lanzar `ollama serve &` en la terminal que se abre con `%xterm`.

In [ ]:
!if ! type ollama > /dev/null; then curl -fsSL https://ollama.com/install.sh | sh; else echo "Ollama ya está instalado."; fi

### Atención
Si no se dispone de una buena GPU en local o en otro servidor, se recomienda correrlo en Colab con una GPU T4.

Recuerda que tras correr la siguiente celda, debes correr el comando `ollama serve &` en la terminal que se habilita.

In [ ]:
%load_ext colabxterm
%xterm

Descargamos el modelo `llama3.2:3b` para usarlo como LLM del asistente de EcoMarket.

In [ ]:
!ollama pull llama3.2:3b

## 3. LLM con LangChain + Ollama
Conectamos LangChain al modelo servido por Ollama.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:3b", validate_model_on_init=True)

llm.invoke("Responde brevemente: ¿qué es EcoMarket?").content

## 4. Construcción del vectorstore (FAISS)

- Dividimos los documentos en chunks para mejorar el retrieval.
- Generamos embeddings con un modelo multilingüe.
- Indexamos todo en FAISS.

In [ ]:
import os
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Chunks generados: {len(chunks)}")

index_path = './faiss_index_ecomarket'
if os.path.exists(index_path):
    vectorstore = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)
else:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(index_path)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

## 5. Prompts del asistente

Los prompts se cargan directamente desde los archivos en la carpeta `prompts/` para mantener una única fuente de verdad.

In [ ]:
from pathlib import Path

PROMPTS_DIR = Path('prompts')

prompt_pedido_tpl = (PROMPTS_DIR / 'prompt_pedido.txt').read_text(encoding='utf-8')
prompt_devolucion_tpl = (PROMPTS_DIR / 'prompt_devolucion.txt').read_text(encoding='utf-8')

print('--- prompt_pedido.txt ---')
print(prompt_pedido_tpl)
print('--- prompt_devolucion.txt ---')
print(prompt_devolucion_tpl)

## 6. Pipeline RAG para EcoMarket

Implementamos un router sencillo que:

1. Decide si la consulta es sobre un **pedido** o sobre **devoluciones**.
2. Recupera el contexto relevante del vectorstore.
3. Aplica el prompt correspondiente (`prompt_pedido.txt` o `prompt_devolucion.txt`).
4. Invoca al LLM y devuelve la respuesta junto con las fuentes.

In [ ]:
import re
from langchain.prompts import PromptTemplate

prompt_pedido = PromptTemplate.from_template(prompt_pedido_tpl)
prompt_devolucion = PromptTemplate.from_template(prompt_devolucion_tpl)

def extraer_tracking(pregunta: str) -> str | None:
    m = re.search(r'\b(\d{3,})\b', pregunta)
    return m.group(1) if m else None

def es_consulta_pedido(pregunta: str) -> bool:
    claves = ['pedido', 'orden', 'tracking', 'envío', 'envio', 'estado']
    return any(k in pregunta.lower() for k in claves) or extraer_tracking(pregunta) is not None

def formatear_contexto(docs):
    return "\n\n".join(d.page_content for d in docs)

def responder(pregunta: str):
    if es_consulta_pedido(pregunta):
        tracking = extraer_tracking(pregunta) or 'N/A'
        docs = retriever.invoke(pregunta)
        docs_pedido = [d for d in docs if d.metadata.get('tipo') == 'pedido'] or docs
        contexto = formatear_contexto(docs_pedido)
        prompt_final = prompt_pedido.format(data=contexto, tracking_number=tracking)
        fuentes = docs_pedido
    else:
        docs = retriever.invoke(pregunta)
        docs_pol = [d for d in docs if d.metadata.get('tipo') == 'politica_devoluciones'] or docs
        contexto = formatear_contexto(docs_pol)
        prompt_final = prompt_devolucion.format(politica=contexto, pregunta=pregunta)
        fuentes = docs_pol

    respuesta = llm.invoke(prompt_final).content
    return {"answer": respuesta, "sources": fuentes}

def imprimir_respuesta(resultado):
    print(resultado['answer'])
    print("\nFuentes:")
    for i, d in enumerate(resultado['sources'], 1):
        src = d.metadata.get('source', 'desconocido')
        tipo = d.metadata.get('tipo', 'N/A')
        extra = d.metadata.get('tracking_number', '')
        etiqueta = f"{tipo}" + (f" #{extra}" if extra else "")
        print(f"- [{i}] {etiqueta} ({src})")

## 7. Preguntas de prueba

In [ ]:
resultado_1 = responder("¿Cuál es el estado de mi pedido 1003?")
imprimir_respuesta(resultado_1)

In [ ]:
resultado_2 = responder("¿Puedo devolver un producto de higiene personal?")
imprimir_respuesta(resultado_2)

## 8. Interfaz conversacional con Gradio

Exponemos el asistente de EcoMarket como un chatbot simple. Cada mensaje se enruta al prompt correspondiente (pedido o devolución) y muestra las fuentes utilizadas.

In [ ]:
import gradio as gr

def format_answer(resultado):
    out = resultado['answer'] + "\n\nFuentes:\n"
    for i, d in enumerate(resultado['sources'], 1):
        tipo = d.metadata.get('tipo', 'N/A')
        extra = d.metadata.get('tracking_number', '')
        etiqueta = tipo + (f" #{extra}" if extra else "")
        out += f"- [{i}] {etiqueta}\n"
    return out

with gr.Blocks() as gr_blocks:
    chatbot = gr.Chatbot()
    msg = gr.Textbox(
        label="¿En qué puedo ayudarte con tu pedido o devolución?",
        placeholder="Escribe tu pregunta aquí y presiona Enter.",
    )
    clear = gr.Button("Limpiar")

    def respond(question, chat_history):
        resultado = responder(question)
        respuesta = format_answer(resultado)
        chat_history.append((question, respuesta))
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

gr_blocks.launch(inline=False)

In [ ]:
gr_blocks.close()

## Conclusiones

- LangChain + Ollama permiten construir un asistente RAG local sin depender de APIs externas.
- FAISS indexa los documentos de EcoMarket (`pedidos.json` y `politica_devoluciones.txt`) y responde con contexto verificable.
- Los prompts se mantienen desacoplados en `prompts/`, lo que facilita iterar sobre el tono y las reglas del asistente sin modificar el código.
- El router simple (pedidos vs. devoluciones) mejora la precisión al aplicar el prompt correcto según la intención del cliente.